In [1]:
from web3 import Web3

RPCS = [
    "https://ethereum-sepolia-rpc.publicnode.com",
    "https://sepolia.drpc.org",
    "https://rpc2.sepolia.org",
]

CONTRACT = "0x488ab4aa772fca36e45e1cb7223f859d2d1cff36"

ABI = [
    {
        "inputs": [{"internalType": "string", "name": "arId", "type": "string"}],
        "name": "getAnchorData",
        "outputs": [{"internalType": "bytes", "name": "", "type": "bytes"}],
        "stateMutability": "view",
        "type": "function"
    },
    {
        "inputs": [{"internalType": "string", "name": "", "type": "string"}],
        "name": "registered",
        "outputs": [{"internalType": "bool", "name": "", "type": "bool"}],
        "stateMutability": "view",
        "type": "function"
    }
]

def get_connection():
    for rpc in RPCS:
        try:
            w3 = Web3(Web3.HTTPProvider(rpc, request_kwargs={"timeout": 5}))
            if w3.is_connected():
                print(f"Connected via: {rpc}")
                return w3
        except Exception as e:
            print(f"Failed {rpc}: {e}")
    raise Exception("All RPCs failed")

def decode_anchor_data(raw_hex, labels):
    data = bytes.fromhex(raw_hex)
    n = len(labels)
    offsets = [int.from_bytes(data[i*32:(i+1)*32], 'big') for i in range(n)]
    for label, offset in zip(labels, offsets):
        length = int.from_bytes(data[offset:offset+32], 'big')
        value = data[offset+32:offset+32+length]
        print(f"  {label}: '{value.decode('utf-8')}'")

w3 = get_connection()
contract = w3.eth.contract(
    address=Web3.to_checksum_address(CONTRACT),
    abi=ABI
)

# CODE anchor
ar_id = "AR-2026-Pvdp0W5"
print(f"\n--- {ar_id} ---")
print(f"registered: {contract.functions.registered(ar_id).call()}")
data = contract.functions.getAnchorData(ar_id).call()
print(f"_anchorData: {len(data)} bytes")
if data:
    decode_anchor_data(data.hex(), ['gitHash','license','language','version','url'])

Connected via: https://ethereum-sepolia-rpc.publicnode.com

--- AR-2026-Pvdp0W5 ---
registered: True
_anchorData: 512 bytes
  gitHash: '18afe3d8ba65ac681791eb1013b57522910244b7'
  license: ''
  language: 'Python'
  version: 'v1.0.0'
  url: 'https://github.com/defipy-devs/defipy'


In [2]:
from web3 import Web3
from web3.types import HexBytes

# Compute topics as HexBytes directly — no string manipulation
ANCHORED_SIG = Web3.keccak(text="Anchored(string,address,uint8,string,string,string,string,string,string)")

def get_ar_id_topic(ar_id):
    return Web3.keccak(text=ar_id)

from eth_abi import decode

RPCS = [
    "https://ethereum-sepolia-rpc.publicnode.com",
    "https://sepolia.drpc.org",
    "https://rpc2.sepolia.org",
]

CONTRACT = "0x488ab4aa772fca36e45e1cb7223f859d2d1cff36"

ABI = [
    {"inputs": [{"internalType": "string", "name": "arId", "type": "string"}],
     "name": "getAnchorData", "outputs": [{"internalType": "bytes", "name": "", "type": "bytes"}],
     "stateMutability": "view", "type": "function"},
    {"inputs": [{"internalType": "string", "name": "", "type": "string"}],
     "name": "registered", "outputs": [{"internalType": "bool", "name": "", "type": "bool"}],
     "stateMutability": "view", "type": "function"},
    {"inputs": [{"internalType": "string", "name": "", "type": "string"}],
     "name": "anchorTypes", "outputs": [{"internalType": "uint8", "name": "", "type": "uint8"}],
     "stateMutability": "view", "type": "function"}
]

FIELD_MAP = {
    0:  ("CODE",       ['gitHash','license','language','version','url']),
    1:  ("RESEARCH",   ['doi','institution','coAuthors','url']),
    2:  ("DATA",       ['dataVersion','format','rowCount','schemaUrl','url']),
    3:  ("MODEL",      ['modelVersion','architecture','parameters','trainingDataset','url']),
    4:  ("AGENT",      ['agentVersion','runtime','capabilities','url']),
    5:  ("MEDIA",      ['mediaType','format','duration','isrc','url']),
    6:  ("TEXT",       ['isbn','publisher','language','url']),
    7:  ("POST",       ['platform','postId','postDate','url']),
    8:  ("ONCHAIN",    ['chainId','assetType','contractAddress','txHash','tokenId','blockNumber','url']),
    9:  ("REPORT",     ['reportType','client','engagement','version','authors','institution','url']),
    10: ("NOTE",       ['noteType','date','participants','url']),
    11: ("EVENT",      ['executor','eventType','eventDate','location','orchestrator','url']),
    12: ("RECEIPT",    ['receiptType','merchant','amount','currency','orderId','platform','url']),
    13: ("LEGAL",      ['docType','jurisdiction','parties','effectiveDate','url']),
    14: ("ENTITY",     ['entityType','entityDomain','verificationMethod','verificationProof','canonicalUrl','documentHash']),
    15: ("PROOF",      ['proofType','proofSystem','circuitId','vkeyHash','auditFirm','auditScope','verifierUrl','reportUrl','proofHash']),
    16: ("RETRACTION", ['reason','replacedBy']),
    17: ("REVIEW",     ['reviewType','evidenceUrl']),
    18: ("VOID",       ['reviewArId','findingUrl','evidence']),
    19: ("AFFIRMED",   ['affirmedBy','findingUrl']),
    20: ("ACCOUNT",    None),
    21: ("OTHER",      ['kind','platform','url','value']),
}

DEPLOY_BLOCK = 10506258

def get_logs_chunked(w3, address, topics, from_block, chunk=50000):
    logs = []
    to_block = w3.eth.block_number
    while from_block <= to_block:
        chunk_end = min(from_block + chunk - 1, to_block)
        logs.extend(w3.eth.get_logs({
            "address": address,
            "topics": topics,
            "fromBlock": from_block,
            "toBlock": chunk_end,
        }))
        from_block = chunk_end + 1
    return logs

    
def decode_strings(data, labels):
    n = len(labels)
    results = {}
    try:
        offsets = [int.from_bytes(data[i*32:(i+1)*32], 'big') for i in range(n)]
        for label, offset in zip(labels, offsets):
            length = int.from_bytes(data[offset:offset+32], 'big')
            results[label] = data[offset+32:offset+32+length].decode('utf-8')
    except Exception as e:
        results['_error'] = str(e)
    return results

def decode_event_data(data_hex):
    """Decode the unindexed portion of the Anchored event:
       (uint8 artifactType, string descriptor, string title, string author,
        string manifestHash, string parentHash, string treeId)
    """
    data = bytes.fromhex(data_hex[2:] if data_hex.startswith('0x') else data_hex)
    decoded = decode(
        ['uint8','string','string','string','string','string','string'],
        data
    )
    labels = ['artifactType','descriptor','title','author','manifestHash','parentHash','treeId']
    return dict(zip(labels, decoded))

def get_connection():
    for rpc in RPCS:
        try:
            w3 = Web3(Web3.HTTPProvider(rpc, request_kwargs={"timeout": 5}))
            if w3.is_connected():
                print(f"Connected via: {rpc}")
                return w3
        except:
            pass
    raise Exception("All RPCs failed")

def inspect_anchor(w3, contract, ar_id):
    print(f"\n{'='*60}")
    print(f"AR-ID: {ar_id}")
    print(f"{'='*60}")

    # ── State checks ──────────────────────────────────────────
    is_reg = contract.functions.registered(ar_id).call()
    print(f"\n[STATE]")
    print(f"  registered : {is_reg}")
    if not is_reg:
        print("  ⚠ NOT REGISTERED")
        return

    type_id = contract.functions.anchorTypes(ar_id).call()
    type_name, labels = FIELD_MAP.get(type_id, ("UNKNOWN", None))
    print(f"  type       : {type_id} ({type_name})")

    # ── Event log — base fields ───────────────────────────────
    print(f"\n[ANCHORED EVENT]")
    ar_id_topic = "0x" + Web3.keccak(text=ar_id).hex()
    logs = get_logs_chunked(
        w3,
        address=Web3.to_checksum_address(CONTRACT),
        topics=[ANCHORED_SIG, get_ar_id_topic(ar_id)],
        from_block=DEPLOY_BLOCK
    )

    if not logs:
        print("  ⚠ No Anchored event found")
    else:
        log = logs[0]
        print(f"  tx         : {log['transactionHash'].hex()}")
        print(f"  block      : {log['blockNumber']}")
        # topic[2] is registrant (indexed address)
        registrant = "0x" + log['topics'][2].hex()[-40:]
        print(f"  registrant : {registrant}")
        event_data = decode_event_data(log['data'].hex())
        for k, v in event_data.items():
            flag = "  ⚠ empty" if not str(v) else ""
            print(f"  {k:<14}: '{v}'{flag}")

    # ── _anchorData — type-specific fields ────────────────────
    print(f"\n[ANCHOR DATA (_anchorData)]")
    data = contract.functions.getAnchorData(ar_id).call()
    print(f"  bytes      : {len(data)}")
    if not data:
        print("  ⚠ EMPTY — extra fields not written")
    elif type_id == 20:
        print(f"  capacity   : {int.from_bytes(data[:32], 'big')}")
    elif labels:
        fields = decode_strings(data, labels)
        for k, v in fields.items():
            flag = "  ⚠ empty" if not v else ""
            print(f"  {k:<20}: '{v}'{flag}")

# ── RUN ───────────────────────────────────────────────────────
w3 = get_connection()
contract = w3.eth.contract(address=Web3.to_checksum_address(CONTRACT), abi=ABI)
# inspect_anchor(w3, contract, "AR-2026-XXXXXXX")  # add more

Connected via: https://ethereum-sepolia-rpc.publicnode.com


In [3]:
inspect_anchor(w3, contract, "AR-2026-Pvdp0W5")


AR-ID: AR-2026-Pvdp0W5

[STATE]
  registered : True
  type       : 0 (CODE)

[ANCHORED EVENT]
  tx         : 0xe36116a1406b366c860772fb2a76f6c685cbf05f4a091ef3cb60c1d0da978646
  block      : 10533679
  registrant : 0xc7a7afde1177fbf0bb265ea5a616d1b8d7ed8c44
  artifactType  : '0'
  descriptor    : 'DeFiPy: Python SDK for DeFi Analytics and Agents'
  title         : 'DeFiPy GitHub'
  author        : 'Ian Moore'
  manifestHash  : '3e2d69872aaeecffb5e6dad0fd8166043f77a29ef67544d5839695a6cdf0da6f'
  parentHash    : ''  ⚠ empty
  treeId        : 'ar-operator-v1'

[ANCHOR DATA (_anchorData)]
  bytes      : 512
  gitHash             : '18afe3d8ba65ac681791eb1013b57522910244b7'
  license             : ''  ⚠ empty
  language            : 'Python'
  version             : 'v1.0.0'
  url                 : 'https://github.com/defipy-devs/defipy'


In [4]:
inspect_anchor(w3, contract, "AR-2026-53mJRwP")


AR-ID: AR-2026-53mJRwP

[STATE]
  registered : True
  type       : 0 (CODE)

[ANCHORED EVENT]
  tx         : 0x70d783ba0a264e519c4ee91a536dbf3eb5ef99966b8c25059d1fbd898536fb8d
  block      : 10535552
  registrant : 0xc7a7afde1177fbf0bb265ea5a616d1b8d7ed8c44
  artifactType  : '0'
  descriptor    : 'UniswapPy: Uniswap V2 / V3 Analytics with Python'
  title         : 'UniswapPy GitHub'
  author        : 'Ian Moore'
  manifestHash  : 'd4c698fd34c7576efc547591eda4560f7a7d696f6a708f601da3fb98b0ef1595'
  parentHash    : 'AR-2026-Pvdp0W5'
  treeId        : 'ar-operator-v1'

[ANCHOR DATA (_anchorData)]
  bytes      : 544
  gitHash             : 'b11a6ef63943fc7437be3a1df444c7986fbf49f7'
  license             : 'Apache-2.0'
  language            : 'Python'
  version             : 'v1.7.0'
  url                 : 'https://github.com/defipy-devs/uniswappy'


In [5]:
inspect_anchor(w3, contract, "AR-2026-Pae2yYP")


AR-ID: AR-2026-Pae2yYP

[STATE]
  registered : True
  type       : 5 (MEDIA)

[ANCHORED EVENT]
  tx         : 0xb87dbad6ba56223fb8fe8f0b83d6e4e6a202ff45bd017c8d92084dc561e461b9
  block      : 10540690
  registrant : 0xc7a7afde1177fbf0bb265ea5a616d1b8d7ed8c44
  artifactType  : '5'
  descriptor    : 'ETHDenver 2024 presentation using DeFiPy'
  title         : 'ETHDenver 2024 Presentation'
  author        : 'Ian Moore'
  manifestHash  : 'd3a2776290d9f10e883546929f05113870cdc1fe2406ffacbff7d3616d137cf3'
  parentHash    : 'AR-2026-Pvdp0W5'
  treeId        : 'ar-operator-v1'

[ANCHOR DATA (_anchorData)]
  bytes      : 416
  mediaType           : ''  ⚠ empty
  format              : ''  ⚠ empty
  duration            : '18:45'
  isrc                : ''  ⚠ empty
  url                 : 'https://www.youtube.com/watch?v=irUM-aB9UFk'
